# 5차시 (2) 프롬프트 엔지니어링과 나만의 챗봇 — 실습

한경국립대 2026 여름특강 · 딥러닝/머신러닝 입문 (초급반)

> **2차시(온라인)** 에서 ChatGPT 같은 **완성된 AI 서비스로** 프롬프트를 체험했죠.
> **오늘(오프라인)** 은 그걸 *서비스로 쓰는 게 아니라* **직접 코드로** 호출하며, 프롬프트가 답을 어떻게 바꾸는지 **화면으로 확인**합니다.

이 노트북에서 우리는 **두 가지**를 직접 해 봅니다.

1. **프롬프트 엔지니어링** — 같은 AI 모델이라도 *어떻게 물어보느냐* 에 따라 답이 크게 달라집니다. 좋은 프롬프트 4원칙을 **코드로 실행하며** 익힙니다.
2. **나만의 Gradio 챗봇** — 코드 몇 줄로 **내 웹 채팅 화면** 을 띄우고, 성격(시스템 프롬프트)과 기억을 넣습니다.

## 준비물 한눈에 보기

- **실행 환경**: Google Colab (이 노트북). 설치할 것은 아래 셀이 알아서 받아 줍니다.
- **OpenAI API 키**: 강의 주최측에서 발급해 드립니다. 키는 **이 노트북 안에서 입력**하며, 코드에 직접 적지 않습니다(보안).
- **이번 노트북에서 쓰는 AI 모델**: `gpt-4o-mini` — 저렴하고 빠른 모델이라 실습에 적합합니다.

순서: ① 라이브러리 설치 → ② API 키 입력 → ③ Part 1 프롬프트 실습 → ④ Part 2 챗봇 만들기.

> ⚠️ **이 노트북의 모델 호출 셀(`ask`·`llm.invoke`·챗봇)은 Colab에서 OpenAI API 키를 입력해야 실행됩니다.** 지금 이 환경에서는 실행되지 않으니, Colab에서 **0-1·0-2 셀을 먼저 실행**한 뒤 아래 셀들을 돌려 보세요.

---
## 0. 환경 세팅

### 0-1. 라이브러리 설치
LangChain(언어 모델을 편하게 부르는 도구)과 Gradio(웹 챗봇 화면 도구)를 설치합니다. **30초~1분** 정도 걸립니다.
설치 후 `pip` 의존성 관련 빨간 경고가 한두 줄 떠도 **무시해도 됩니다**(실행에는 지장 없음).

In [5]:
# -qU : 조용히(-q) + 최신 버전으로 업그레이드(-U) 설치
!pip install -qU langchain-openai langchain-core
!pip install -qU "gradio>=6"
print("설치 완료! 위에 경고가 보여도 괜찮습니다.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 120.4/120.4 kB 5.6 MB/s eta 0:00:00
설치 완료! 위에 경고가 보여도 괜찮습니다.


### 0-1½. 잠깐 — API? API 키? (처음이라면 꼭 읽어요)

지금부터 우리는 OpenAI의 AI 모델을 **코드로 불러다 씁니다.** 그 창구가 **API** 예요.

- **API** = 남이 만든 강력한 기능(여기선 AI 모델)을 **내 코드에서 불러 쓰는 창구**. 예를 들어 **카카오 로그인·지도(카카오 API)**, **네이버 지도·검색(네이버 API)** 도 이렇게 씁니다. → API를 알면 남의 서비스 기능을 내 프로그램에 붙일 수 있어요.
- **API 키** = 그 창구를 여는 **내 비밀번호이자 결제 수단**. 이 키로 쓴 만큼 **내 계정에 요금이 청구** 됩니다.

**⚠️ API 키는 절대 남에게 보이면 안 됩니다.**
- GitHub·블로그·노트북에 키를 그대로 올렸다가 **몇 시간 만에 수천 달러가 청구된** 사고가 흔합니다(학생·개인 개발자 피해 다수). 유출된 키는 봇이 몇 분 만에 찾아내 마구 씁니다.
- 그래서 아래 셀은 키를 화면·코드에 남기지 않는 **`getpass`(가림 입력)** 를 씁니다. **키를 코드에 직접 붙여넣지 마세요.**

> 더 보기(수업 중 화면으로): OpenAI 계정 보안 가이드 https://help.openai.com/en/articles/8304786-how-can-i-keep-my-openai-accounts-secure · 유출 사고 기사(8,000+ 키 노출) https://www.vicarius.io/articles/8-000-chatgpt-api-keys-exposed-across-github-production-sites · 카카오 개발자 https://developers.kakao.com · 네이버 지도 API https://www.ncloud.com/product/applicationService/maps

### 0-2. OpenAI API 키 입력
아래 셀을 실행하면 입력창이 나옵니다. **주최측에서 받은 키를 붙여넣고 Enter** 를 누르세요.
- `getpass` 를 쓰기 때문에 입력한 키는 **화면에 보이지 않습니다**(어깨너머로 노출되지 않게).
- 키는 `OPENAI_API_KEY` 라는 환경변수에 저장되어, 아래 모든 셀이 자동으로 사용합니다.
- 런타임을 다시 시작(연결 끊김)하면 이 셀을 **다시 실행**해야 합니다.

In [6]:
import os
import getpass

# 이미 입력했다면 다시 묻지 않습니다.
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API 키를 입력하세요: ")

print("API 키가 설정되었습니다." if os.environ.get("OPENAI_API_KEY") else "키가 비어 있습니다. 다시 실행하세요.")

API 키가 설정되었습니다.


### 0-3. 모델 한 번 불러보기 (연결 확인)
키가 잘 설정됐는지, 모델이 실제로 답하는지 **가장 간단한 한 줄**로 확인합니다.

- `ChatOpenAI` : OpenAI의 채팅 모델을 부르는 객체.
- `temperature` : 답변의 *무작위성/창의성*. 0에 가까울수록 매번 비슷하고 딱딱하게, 1에 가까울수록 다양하고 자유롭게 답합니다.
- `.invoke("...")` : 모델에게 메시지를 보내고 답을 받아오는 함수.

In [7]:
from langchain_openai import ChatOpenAI

# 이번 실습 내내 쓸 모델. 저렴하고 빠른 gpt-4o-mini 사용.
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

answer = llm.invoke("한 문장으로 자기소개를 해줘.")
print(answer.content)

안녕하세요, 저는 다양한 정보를 제공하고 질문에 답변하는 AI입니다.


---
# Part 1 · 프롬프트 엔지니어링

**프롬프트(prompt)** = 우리가 AI에게 건네는 *지시문/질문* 입니다.
모델은 그대로 두고 **프롬프트만 잘 바꿔도** 답의 품질이 확 올라갑니다. 이것을 다듬는 일이 **프롬프트 엔지니어링** 입니다.

아래 4가지 방법을 *나쁜 예 → 좋은 예* 로 직접 비교해 봅니다.

1. **명료하게 지시하기** — 구체적으로, 빠짐없이 요청
2. **참고 텍스트 제공하기** — 답의 근거가 될 자료를 함께 줌
3. **작업을 작게 쪼개기** — 큰 일을 단계로 나눠 시킴
4. **페르소나(역할) 부여** — "너는 ○○야" 로 말투·전문성 지정

> 참고: OpenAI 공식 프롬프트 가이드 — https://developers.openai.com/api/docs/guides/prompt-engineering

먼저, 비교를 편하게 하려고 **프롬프트를 넣으면 답을 출력하는 작은 함수** 를 하나 만들어 둡니다.

In [9]:
def ask(prompt, temperature=0.7):
    """프롬프트(문자열)를 모델에 보내고 답변 텍스트를 반환 + 화면에 출력."""
    model = ChatOpenAI(model="gpt-4o-mini", temperature=temperature)
    result = model.invoke(prompt)
    print(result.content)
    return result.content

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 연습 0 · 비교 함수 <code>compare(a, b)</code> 직접 만들기<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성 · 가이드</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>왜</b> &nbsp; 이번 Part 내내 <b>'나쁜 프롬프트 vs 좋은 프롬프트'</b>를 나란히 비교합니다. 매번 <code>ask()</code>를 두 번 치는 대신, <b>두 프롬프트를 받아 한꺼번에 보여주는 함수</b>를 직접 만들어 재사용합니다.<br><b>문제</b> &nbsp; 아래 <code>compare(a, b)</code> 뼈대의 빈 곳(두 군데)을 채워, 프롬프트 <code>a</code>·<code>b</code>를 각각 <code>ask()</code>로 호출해 <b>구분선과 함께 위/아래로</b> 출력하세요.<br><b>힌트</b> &nbsp; 위에서 만든 <code>ask()</code>는 답을 <b>화면에 출력하면서 반환</b>합니다. 각 프롬프트 앞에 어떤 쪽인지 라벨을 <code>print</code> 하고 <code>ask()</code>를 호출하면 됩니다.<br><b>예상 동작</b> &nbsp; <code>compare("고양이 알려줘", "고양이 특징을 3가지로 짧게 알려줘")</code>를 실행하면 두 답이 <b>[A] / [B]</b> 로 나뉘어 나옵니다.
</div>
</div>

In [12]:
# 🔧 직접 작성 (가이드형) — 뼈대는 주어졌으니 빈 곳(아래 두 군데)만 채우세요.
def compare(a, b):
    """두 프롬프트 a, b 를 각각 ask() 로 호출해, 구분선과 함께 나란히 출력."""
    print(ask(a))
    # TODO: 프롬프트 a 를 ask() 로 호출하세요 (ask 는 출력까지 해 줍니다)

    print(ask(b))
    # TODO: 프롬프트 b 를 ask() 로 호출하세요


# 완성했다면 아래로 시험해 보세요.
compare("고양이에 대해 알려줘.",
        "고양이를 처음 키우는 사람을 위해 고양이의 특징 3가지를 짧게 알려줘.")

고양이는 집에서 기르는 가장 인기 있는 애완동물 중 하나로, 과학적으로는 Felis catus라고 불립니다. 고양이는 작은 포유류로, 일반적으로 귀여운 외모와 독립적인 성격으로 알려져 있습니다. 이들은 주로 육식성으로, 작은 설치류, 조류, 곤충 등을 먹으며, 뛰어난 사냥 능력을 가지고 있습니다.

고양이는 다양한 품종이 있으며, 각 품종마다 고유한 외모와 성격이 있습니다. 일반적인 고양이의 특징으로는 부드러운 털, 날카로운 발톱, 그리고 뛰어난 균형 감각이 있습니다. 또한, 고양이는 매우 민감한 감각을 가지고 있어, 눈은 어두운 곳에서도 잘 볼 수 있고, 귀는 높은 주파수의 소리를 감지하는 데 능숙합니다.

고양이는 매우 독립적인 동물로 알려져 있지만, 주인과의 유대 관계도 중요하게 생각합니다. 그들은 종종 애정 표현을 위해 몸을 비비거나, 주인의 무릎에 앉는 등의 행동을 합니다. 또한, 고양이는 자주 고양이 소리를 내며, 이는 의사소통의 한 방법입니다.

고양이는 일반적으로 건강한 동물이지만, 정기적인 건강 검진과 예방 접종이 필요합니다. 또한, 실내에서 기르는 경우에는 적절한 운동과 자극을 제공해 주는 것이 중요합니다. 고양이는 장기적인 동반자로서, 많은 사람들에게 사랑받고 있습니다.
고양이는 집에서 기르는 가장 인기 있는 애완동물 중 하나로, 과학적으로는 Felis catus라고 불립니다. 고양이는 작은 포유류로, 일반적으로 귀여운 외모와 독립적인 성격으로 알려져 있습니다. 이들은 주로 육식성으로, 작은 설치류, 조류, 곤충 등을 먹으며, 뛰어난 사냥 능력을 가지고 있습니다.

고양이는 다양한 품종이 있으며, 각 품종마다 고유한 외모와 성격이 있습니다. 일반적인 고양이의 특징으로는 부드러운 털, 날카로운 발톱, 그리고 뛰어난 균형 감각이 있습니다. 또한, 고양이는 매우 민감한 감각을 가지고 있어, 눈은 어두운 곳에서도 잘 볼 수 있고, 귀는 높은 주파수의 소리를 감지하는 데 능숙합니다.

고양이는 매우 독립적인 동물로 알려져 있지만, 주인과의 유대 관계

### 0-4. 잠깐 — LangChain? RAG? (처음 듣는 용어 정리)

- **LangChain**: AI 모델·프롬프트·도구·기억 같은 조각을 **레고블록처럼 이어 붙여** AI 앱을 쉽게 만드는 파이썬 라이브러리. 방금 쓴 `ChatOpenAI` 도 그 블록 하나예요. (오늘 챗봇을, 다음엔 RAG·에이전트를 이걸로 만듭니다.)
- **RAG**(Retrieval-Augmented Generation): AI가 답하기 전에 **내 문서에서 관련 부분을 찾아(검색) 근거로 답** 하게 하는 방법 → **다음 노트북(s5-03)** 에서 직접 만듭니다.

> 더 보기(수업 중 화면으로): LangChain 공식 문서 https://docs.langchain.com/oss/python/learn

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 온램프 · 값 하나만 바꿔 관찰하기<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">관찰</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래 코드는 이미 실행됩니다. <code>temperature</code> 값만 <b>0 → 1.3</b> 으로 바꿔 같은 프롬프트를 여러 번 실행해 보세요.<br><b>힌트</b> &nbsp; <code>ask(프롬프트, temperature=숫자)</code> — 숫자는 0~2. 0에 가까우면 매번 비슷하게, 커질수록 자유롭게 답합니다.<br><b>관찰 포인트</b> &nbsp; <code>temperature=0</code> 은 여러 번 돌려도 답이 거의 같고, <code>1.3</code> 은 매번 표현이 달라지면 성공.
</div>
</div>

In [13]:
# temperature 만 바꿔 같은 프롬프트를 여러 번 실행 → 답의 '일관성 vs 다양성' 관찰
prompt = "가을을 주제로 짧은 한 줄 카피를 써줘."

print("===== temperature = 0 (딱딱·일관) =====")
for i in range(2):
    ask(prompt, temperature=0)

print("\n===== temperature = 1.3 (자유·다양) =====")   # 이 숫자만 바꿔 보세요!
for i in range(2):
    ask(prompt, temperature=1.3)

===== temperature = 0 (딱딱·일관) =====
"가을의 바람 속에, 추억이 물들어 간다."
"가을, 낙엽 속에 숨겨진 추억의 향기."

===== temperature = 1.3 (자유·다양) =====
“가을, 바람에 실린 나뭇잎의 속삭임 속으로 가슴을 열다.”
"가을, 바람에 실려온 추억의 향기."


## 1-1. 명료하게 지시하기
모호한 질문은 모호한 답을 부릅니다. **무엇을, 어떤 형식으로, 어떤 조건으로** 원하는지 구체적으로 적으면 답이 좋아집니다.

아래 두 셀을 각각 실행해 **답의 차이**를 비교해 보세요.

In [14]:
# (나쁜 예) 너무 막연한 질문
print("===== 모호한 프롬프트 =====")
ask("피보나치 수열 코드 짜줘.")

===== 모호한 프롬프트 =====
피보나치 수열을 생성하는 코드는 여러 가지 방법으로 구현할 수 있습니다. 아래는 Python을 사용한 간단한 예제입니다. 이 코드는 재귀적으로 피보나치 수열의 n번째 항을 계산하는 방법과 반복문을 사용하는 방법 두 가지를 보여줍니다.

### 1. 재귀적 방법

```python
def fibonacci_recursive(n):
    if n <= 0:
        return 0
    elif n == 1:
        return 1
    else:
        return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)

# 사용 예
n = 10  # 피보나치 수열의 10번째 항
print(f"피보나치 수열의 {n}번째 항: {fibonacci_recursive(n)}")
```

### 2. 반복문을 사용하는 방법

```python
def fibonacci_iterative(n):
    a, b = 0, 1
    for _ in range(n):
        a, b = b, a + b
    return a

# 사용 예
n = 10  # 피보나치 수열의 10번째 항
print(f"피보나치 수열의 {n}번째 항: {fibonacci_iterative(n)}")
```

### 3. 피보나치 수열의 첫 n개 항 출력하기

```python
def fibonacci_sequence(n):
    sequence = []
    a, b = 0, 1
    for _ in range(n):
        sequence.append(a)
        a, b = b, a + b
    return sequence

# 사용 예
n = 10  # 피보나치 수열의 처음 10개 항
print(f"피보나치 수열의 처음 {n}개 항: {fibonacci_sequence(n)}")
```

위의 코드 중에서 원하는 방식으로 피보나치 수열을 계산하는 기능을 사용

'피보나치 수열을 생성하는 코드는 여러 가지 방법으로 구현할 수 있습니다. 아래는 Python을 사용한 간단한 예제입니다. 이 코드는 재귀적으로 피보나치 수열의 n번째 항을 계산하는 방법과 반복문을 사용하는 방법 두 가지를 보여줍니다.\n\n### 1. 재귀적 방법\n\n```python\ndef fibonacci_recursive(n):\n    if n <= 0:\n        return 0\n    elif n == 1:\n        return 1\n    else:\n        return fibonacci_recursive(n - 1) + fibonacci_recursive(n - 2)\n\n# 사용 예\nn = 10  # 피보나치 수열의 10번째 항\nprint(f"피보나치 수열의 {n}번째 항: {fibonacci_recursive(n)}")\n```\n\n### 2. 반복문을 사용하는 방법\n\n```python\ndef fibonacci_iterative(n):\n    a, b = 0, 1\n    for _ in range(n):\n        a, b = b, a + b\n    return a\n\n# 사용 예\nn = 10  # 피보나치 수열의 10번째 항\nprint(f"피보나치 수열의 {n}번째 항: {fibonacci_iterative(n)}")\n```\n\n### 3. 피보나치 수열의 첫 n개 항 출력하기\n\n```python\ndef fibonacci_sequence(n):\n    sequence = []\n    a, b = 0, 1\n    for _ in range(n):\n        sequence.append(a)\n        a, b = b, a + b\n    return sequence\n\n# 사용 예\nn = 10  # 피보나치 수열의 처음 10개 항\nprint(f"피보나치 수열의 처음 {n}개 항: {fibonacci_sequence(n)}")\n```\n\n위의 코드 중에서 원

In [ ]:
# (좋은 예) 언어, 조건, 형식, 설명까지 구체적으로
print("===== 구체적인 프롬프트 =====")
ask(
    "파이썬으로 피보나치 수열의 n번째 값을 구하는 함수를 작성해줘. "
    "조건: ① 반복문 사용(재귀 금지), ② 입력이 음수면 안내 메시지 출력, "
    "③ 각 줄에 무엇을 하는지 한국어 주석을 달아줘."
)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-1 · 나만의 '모호 vs 구체' 프롬프트<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 여러분이 궁금한 주제로 <b>(모호한 질문)</b> 과 <b>(구체적인 질문)</b> 을 한 쌍 만들어, 방금 만든 <code>compare()</code> 로 비교하세요.<br><b>힌트</b> &nbsp; 구체 프롬프트엔 ① <b>무엇을</b> ② <b>어떤 형식으로</b>(표·3줄·코드 등) ③ <b>어떤 조건</b>(글자 수·대상 독자)을 넣습니다. 주제 예: 여행 코스·이메일·요약.<br><b>관찰 포인트</b> &nbsp; 구체 프롬프트 쪽이 형식·분량이 내 요구에 맞으면 성공.
</div>
</div>

In [ ]:
# ①모호 ②구체 프롬프트를 직접 채워, compare() 로 한 번에 비교
vague_prompt = "____"      # 예: "여행 코스 추천해줘."
clear_prompt = "____"      # 예: "부산 1박 2일 코스를, 오전/오후/저녁 표로, 이동시간 포함해 추천해줘."

compare(vague_prompt, clear_prompt)

## 1-2. 참고할 텍스트를 함께 제공하기
모델은 *모르는 사실*(최신 정보, 우리 회사 내부 자료 등)은 지어낼 수 있습니다(이를 **환각, hallucination** 이라 합니다).
이때 **답의 근거가 될 텍스트를 프롬프트에 같이 넣어** 주고 "이 내용만 보고 답하라"고 지시하면 훨씬 정확해집니다.

> 이 아이디어를 문서 전체로 확장한 것이 바로 **다음 노트북의 RAG** 입니다.

In [ ]:
# 모델이 학습 시점에 몰랐을 법한 '가상의 제품 설명'을 직접 제공
reference = """
[제품 안내문 — 한경 스마트텀블러 X2]
- 용량: 500ml
- 보온: 12시간 / 보냉: 24시간
- 무게: 320g
- 색상: 미드나잇 블랙, 아이보리, 포레스트 그린
- 가격: 32,000원
- 보증: 구매일로부터 1년
"""

question = "이 텀블러 보냉은 몇 시간이고 가격은 얼마야? 안내문에 없는 내용은 '안내문에 없음'이라고 답해줘."

prompt = f"""아래 안내문만 참고해서 질문에 답해줘.

안내문:
{reference}

질문: {question}"""

ask(prompt, temperature=0)   # 사실 질문이므로 temperature=0(딱딱하지만 일관됨)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-2 · 근거에 없는 걸 물어보면?<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">관찰</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 위 <code>reference</code> 안내문에 <b>없는</b> 내용(예: 방수·식기세척기·A/S 전화번호)을 물어, 모델이 <b>'안내문에 없음'</b>이라고 답하는지 확인하세요.<br><b>힌트</b> &nbsp; 아래 <code>new_question</code> 에 안내문에 안 적힌 항목을 한 가지 적으면 됩니다. <code>reference</code> 변수는 그대로 씁니다.<br><b>관찰 포인트</b> &nbsp; 지어내지 않고 '안내문에 없음'이라고 답하면 성공 — 환각을 줄이는 핵심.
</div>
</div>

In [ ]:
# TODO: 안내문에 없는 질문을 넣어 '안내문에 없음'이라고 답하는지 확인
new_question = ""  # 예: "이 텀블러는 식기세척기를 써도 되나요?"
prompt = f"""아래 안내문만 참고해서 질문에 답해줘. 없는 내용은 '안내문에 없음'이라고 답해.

안내문:
{reference}

질문: {new_question}"""
ask(prompt, temperature=0)

## 1-3. 복잡한 작업은 작은 단계로 쪼개기
큰 요청을 한 번에 던지면 빠뜨리는 부분이 생깁니다. **순서가 있는 단계(1·2·3…)로 나눠** 지시하면 결과가 더 완성도 높고 일관됩니다.

아래는 "공 튀기기 게임 만들기"를 *한 줄 요청* vs *단계별 요청* 으로 비교합니다.
(게임 코드를 Colab에서 실행하진 않습니다 — **생성된 코드의 완성도 차이**만 비교하세요.)

In [ ]:
# (나쁜 예) 한 줄 요청
print("===== 한 줄 요청 =====")
ask("파이썬으로 공 튀기기 게임 코드 만들어줘.")

In [ ]:
# (좋은 예) 단계로 쪼갠 요청
print("===== 단계별 요청 =====")
ask(
    "파이썬 pygame으로 '공 튀기기 게임'을 아래 단계 순서대로 빠짐없이 구현해줘.\n"
    "1) 게임 창(800x600) 생성\n"
    "2) 원형 공 객체 정의(초기 위치·속도 포함)\n"
    "3) 공이 벽에 부딪히면 튕기게 만들기\n"
    "4) 게임 루프에서 매 프레임 위치 갱신·화면 다시 그리기\n"
    "5) 방향키로 공의 속도를 바꾸기\n"
    "6) 위 1~5를 합친 전체 코드를 한 번에 제시"
)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-3 · 내 요청을 '단계'로 쪼개기<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 평소 하고 싶은 요청 하나를 <b>(한 줄 요청)</b> 과 <b>(단계로 쪼갠 요청)</b> 으로 각각 만들어 실행하고, 결과의 완성도를 비교하세요.<br><b>힌트</b> &nbsp; 큰 일을 <code>1) … 2) … 3) …</code> 순서로 나누고 마지막에 "위를 합쳐 한 번에 제시"를 덧붙입니다. 주제 예: 블로그 글·미니 게임·여행 계획.<br><b>관찰 포인트</b> &nbsp; 단계로 쪼갠 쪽이 빠뜨림 없이 더 완성도 높으면 성공.
</div>
</div>

In [ ]:
# 같은 목표를 (한 줄) vs (단계 분할) 로 요청해 비교
oneline_prompt = "____"    # 예: "여행 블로그 글 하나 써줘."
steps_prompt = "____"      # 예: "부산 여행 블로그 글을 아래 순서로 써줘.\n1) 제목\n2) 도입\n3) 일정 3개\n4) 마무리\n5) 위를 합쳐 완성글 제시"

print("===== 한 줄 요청 =====")
ask(oneline_prompt)
print("\n===== 단계별 요청 =====")
ask(steps_prompt)

## 1-4. 페르소나(역할) 부여하기
"너는 ○○야" 라고 **역할을 정해** 주면 말투·난이도·전문성이 그 역할에 맞게 바뀝니다.
같은 질문을 **두 가지 페르소나**에게 던져 답이 어떻게 달라지는지 비교해 봅시다.

In [ ]:
question = "벡터(vector)가 뭔지 설명해줘."

print("===== 페르소나 A: 초등학생에게 설명하는 친절한 선생님 =====")
ask(f"너는 초등학생에게 설명하는 친절한 선생님이야. 쉬운 비유와 이모지를 써서 설명해줘.\n질문: {question}")

print("\n" + "=" * 50 + "\n")

print("===== 페르소나 B: 대학생에게 설명하는 데이터 사이언스 전문가 =====")
ask(f"너는 데이터 사이언스 전문가야. 대학생에게 수학적 정의를 포함해 전문적으로 설명해줘.\n질문: {question}")

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-4 · 페르소나 반복문 직접 작성<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 여러 페르소나가 담긴 <code>personas</code> 리스트를 <b>for 로 돌며</b>, 같은 질문을 각 페르소나로 던지는 반복문을 직접 작성하세요.<br><b>힌트</b> &nbsp; <code>for persona in personas:</code> 안에서 프롬프트를 조립(페르소나 + 줄바꿈 + "질문: " + question)해 <code>ask()</code> 를 호출합니다. 구분선/라벨을 함께 출력하면 보기 좋아요.<br><b>예상 결과</b> &nbsp; 같은 질문인데 페르소나마다 어휘·길이·태도가 확 달라지면 성공.
</div>
</div>

In [ ]:
# 준비물(제공): 같은 질문 + 여러 페르소나
question = "환불하려면 어떻게 해야 하나요?"
personas = [
    "너는 친절한 고객센터 상담원이야. 단계별로 안내해줘.",
    "너는 무뚝뚝한 편의점 선배야. 짧고 툭툭 대답해줘.",
    "너는 신난 8살 아이야. 쉬운 말로 신나게 답해줘.",
]

# 🔧 직접 작성 (스켈레톤) — for 뼈대는 주어졌습니다. 안쪽 두 줄을 채우세요.
for persona in personas:
    print("=" * 45)
    print("[페르소나]", persona)
    prompt = ____        # persona 문장 + 줄바꿈 + "질문: " + question 을 한 문자열로
    ____                 # 위 prompt 를 모델에 보내 답을 출력 (ask 는 출력까지 해 줌)


## 1-5. 프롬프트가 정답을 바꾼다 — "단계별로 생각해"의 힘 (+ 모델 크기)

같은 문제·같은 모델이라도 **프롬프트를 어떻게 쓰느냐**에 따라 답이 달라집니다.
아래 계산 문제를 ① **모호하게**("숫자만 답해") ② **단계별로 생각하게**(CoT) 물어보고, 나아가 **작은 모델 vs 큰 모델**로도 비교해 봅니다.

> 문제: 가게에 연필이 **5다스** 있었다. 그중 **3다스**를 팔았고, 남은 연필의 **1/4** 만큼을 새로 들여왔다. 지금 연필은 몇 자루? (정답: **30자루**)

In [ ]:
# 비교할 두 모델 (※ 계정에서 호출 가능한 이름으로 바꿔도 됩니다)
SMALL_MODEL = "gpt-4o-mini"   # 작은·저렴한 모델
LARGE_MODEL = "gpt-4o"        # 큰·강한 모델

def ask_with(model_name, prompt, temperature=0):
    """지정한 모델로 프롬프트를 보내고 답변 텍스트를 반환."""
    model = ChatOpenAI(model=model_name, temperature=temperature)
    return model.invoke(prompt).content

In [ ]:
problem = "가게에 연필이 5다스 있었다. 그중 3다스를 팔았고, 남은 연필의 1/4만큼을 새로 들여왔다. 지금 연필은 몇 자루인가?"

# 1) 모호한 프롬프트 - 바로 답만
vague_prompt = problem + " 숫자만 답해."

# 2) 단계별로 생각하게 (CoT) - 계산 과정을 쓰고 답
cot_prompt = problem + " 한 단계씩 계산 과정을 쓰고, 마지막 줄에 '답: N자루' 형식으로 답해."

In [ ]:
print("========== (1) 모호한 프롬프트 (작은 모델) ==========")
for i in range(3):                      # 여러 번 = 답이 들쭉날쭉한지 확인
    print(f"{i+1}회:", ask_with(SMALL_MODEL, vague_prompt))

print("\n========== (2) 단계별(CoT) 프롬프트 (작은 모델) ==========")
print(ask_with(SMALL_MODEL, cot_prompt))

print("\n========== (3) 모호한 프롬프트 (큰 모델) ==========")
print(ask_with(LARGE_MODEL, vague_prompt))

**관찰 포인트**

- **(1) 모호 + 작은 모델**: 답이 `18` · `36` · `30` 처럼 **매번 다르고 자주 틀립니다**. 추론 없이 바로 찍기 때문이죠.
- **(2) 단계별(CoT) + 작은 모델**: `60 → 24 → +6 → 30` 처럼 과정을 밟아 **안정적으로 정답(30)** 을 냅니다. → **모델은 그대로, 프롬프트 한 마디로 정답률이 오른다.**
- **(3) 모호 + 큰 모델**: 큰 모델은 모호하게 물어도 대체로 맞힙니다. → **작은·저렴한 모델일수록 좋은 프롬프트가 더 중요**(프롬프트 = 성능·비용 절감).

> 응용: `cot_prompt` 를 다른 계산·논리 문제로 바꿔, "단계별로 생각해"가 있을 때와 없을 때를 직접 비교해 보세요.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-5 · CoT 답을 <b>코드로 채점</b>하기<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; 아래처럼 (바로 답) vs (단계별 CoT) 두 응답을 받은 뒤, 각 응답에서 <b>최종 숫자</b>를 뽑아 <code>correct_answer</code>와 비교해 <b>맞음/틀림을 코드로 판정</b>하세요(맨눈 채점 X).<br><b>힌트</b> &nbsp; <code>import re</code> · <code>re.findall(r'-?\d+', 응답)</code>로 숫자를 모두 뽑아 마지막 값(<code>[-1]</code>)을 <code>int</code>로 → <code>correct_answer</code>와 <code>==</code> 비교.<br><b>예상 동작</b> &nbsp; 단계별(CoT) 쪽이 더 자주 정답을 맞히면 성공 — 특히 작은 모델에서.
</div>
</div>

In [ ]:
# ✏️ 연습 1-5 · CoT 답을 '코드로 채점'하기
# 준비물(제공): 헷갈리는 문제 하나 + 정답
grade_problem = "가게에 사탕이 4묶음 있고 한 묶음은 12개다. 그중 15개를 팔았다. 남은 사탕은 몇 개인가?"
correct_answer = 33   # 정답 (4 x 12 - 15 = 33)

# 두 방식으로 답을 받는다 (바로 답 vs 단계별 CoT)
plain_reply = ask_with(SMALL_MODEL, grade_problem + " 다른 말 없이 숫자만 답해.")
cot_reply   = ask_with(SMALL_MODEL, grade_problem + " 한 단계씩 계산 과정을 쓰고, 마지막 줄에 '답: N' 형식으로 답해.")

print("[바로 답]", plain_reply)
print("[단계별 ]", cot_reply[-80:])   # 뒷부분(마지막 답)만

import re

# 🔧 직접 작성 (스켈레톤) — 응답에서 '최종 숫자'를 뽑아 정답과 비교하세요.
def extract_number(text):
    nums = ____                              # text 안의 모든 정수를 re 로 찾기 (결과는 리스트)
    return int(nums[-1]) if nums else None   # (뼈대) 마지막 숫자를 최종 답으로 본다

for label, reply in [("바로 답", plain_reply), ("단계별 CoT", cot_reply)]:
    pred = ____                              # extract_number 로 reply 에서 최종 숫자 뽑기
    ok = ____                                # pred 가 correct_answer 와 같은지 (True/False)
    print(f"{label}: 예측={pred} / 정답={correct_answer} -> {'O 맞음' if ok else 'X 틀림'}")


## 1-6. JSON으로 구조화해서 뽑고, 코드로 파싱하기

프롬프트로 **정해진 형식(JSON)** 만 뽑게 하면, 그 결과를 **프로그램이 바로 읽어** 쓸 수 있습니다.
리뷰 한 문장에서 감성과 이유를 `{"sentiment": ..., "reason": ...}` 형태로 뽑고, `json.loads` 로 **파싱해 필드를 꺼내** 봅니다. 파싱이 실패하면 프롬프트를 고쳐 다시 — 이게 **신뢰할 수 있는 출력** 을 만드는 프롬프트 엔지니어링입니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-6 · JSON만 뽑아 <code>json.loads</code>로 파싱<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; ① <code>review</code>의 감성·이유를 <code>{"sentiment": ..., "reason": ...}</code> <b>JSON만</b> 출력하는 프롬프트를 쓰고, ② 응답을 <code>json.loads</code>로 파싱해 두 필드를 꺼내 출력하세요.<br><b>힌트</b> &nbsp; 프롬프트에 <b>"설명·코드블록 없이 JSON만"</b>을 명시해야 파싱이 쉽습니다. 파싱은 <code>import json</code> → <code>data = json.loads(응답)</code> → <code>data["sentiment"]</code>.<br><b>예상 동작</b> &nbsp; 에러 없이 감성·이유가 각각 출력되면 성공. 파싱 에러가 나면 프롬프트를 고쳐 다시 실행.
</div>
</div>

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">⚠️ 흔한 실수 · "JSON으로 답해줘"라고만 하면 — 결과를 예측해 보세요</div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>상황</b> &nbsp; 리뷰 감성을 <b>코드로 자동 처리</b>하려고 모델에게 JSON으로 답하라 했습니다. 그리고 그 답을 <code>json.loads()</code> 로 파싱합니다.<br><b>예측</b> &nbsp; 모델은 순수 JSON만 줄까요? <code>json.loads()</code> 가 성공할까요?
</div>
</div>

In [ ]:
# ⚠️ 흔한 실수 — 'JSON만 출력'을 안 박으면 코드가 깨진다
import json
review_demo = '배송은 빨랐지만 포장이 부실했어요.'
reply = ask('리뷰 감성을 JSON으로 분석해줘: ' + review_demo)   # ask는 응답을 출력+반환
data = json.loads(reply)   # ← 응답에 ```json 코드펜스가 붙어 있으면 여기서 터진다!
print(data)

<details>
<summary><b>🔑 왜 이렇게 되나 · 올바른 코드</b></summary>

모델은 친절하게 <code>```json ... ```</code> 코드펜스나 설명을 덧붙이곤 합니다. 사람 눈엔 JSON이지만 <code>json.loads()</code> 에겐 <b>맨 앞 백틱부터 문법 오류</b>라 터져요. 프롬프트에 <b>'설명·코드블록 없이 JSON만 출력'</b>을 못 박고, 그래도 대비해 펜스를 벗겨 파싱하세요.

```python
reply = ask('리뷰 감성을 분석해 JSON만 출력해. 설명·코드블록 없이. 리뷰: ' + review_demo)
cleaned = reply.strip().strip('`')            # 혹시 남은 백틱 제거
cleaned = cleaned.replace('json', '', 1) if cleaned.startswith('json') else cleaned
data = json.loads(cleaned)
```

👉 <b>교훈</b> : 모델 출력을 <b>코드로 받을 땐 형식을 계약처럼</b> 못 박고, 파싱은 방어적으로.

</details>

In [ ]:
# ✏️ 연습 1-6 · JSON으로 뽑고 코드로 파싱하기
import json

review = "배송은 빨랐지만 포장이 부실해서 모서리가 찌그러져 왔어요. 제품 자체는 만족합니다."

# 🔧 직접 작성 (프롬프트) — review 의 감성·이유를 'JSON만' 출력하도록 프롬프트를 쓰세요.
#   목표 형태: {"sentiment": "긍정" 또는 "부정", "reason": "한 줄 이유"}
prompt = ""   # 여기에 직접 작성

reply = ask_with(SMALL_MODEL, prompt)
print("모델 원본 응답:", reply)

# 🔧 직접 작성 (스켈레톤) — 아래 세 칸을 채워 응답을 파싱·출력하세요.
data = ____                  # reply(문자열) 를 파이썬 dict 로 바꾸기
print("감성:", ____)          # dict 에서 sentiment 값 꺼내기
print("이유:", ____)          # dict 에서 reason 값 꺼내기
#   파싱이 실패(에러)하면 위 프롬프트를 고쳐 다시 실행(= 신뢰성 있는 출력 만들기).


## 1-7. 학습 없이 분류하고, 정확도를 코드로 채점하기

s5-01의 BERT는 **데이터로 학습**해서 분류했지만, 여기서는 **학습 없이 프롬프트만으로** 분류합니다(이것을 zero-shot 이라고 합니다).
(문장, 정답) 5개를 반복문으로 돌며 예측→정답 비교→**정확도** 를 코드로 계산해, 프롬프트 분류기가 얼마나 맞히는지 **수치로** 확인합니다.

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 연습 1-7 · zero-shot 분류 + <b>정확도</b> 채점<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; ① 문장을 <b>긍정/부정 중 한 단어</b>로만 답하게 하는 프롬프트로 <code>classify()</code>를 완성하고, ② <code>samples</code> 5개를 반복문으로 예측→정답 비교해 <b>맞은 수와 정확도</b>를 출력하세요.<br><b>힌트</b> &nbsp; <code>ask_with(SMALL_MODEL, 프롬프트).strip()</code> 로 답만 받고, <code>for 문장, 정답 in samples:</code> 안에서 <code>pred == 정답</code> 이면 개수를 세어 <code>맞은수 / len(samples)</code>.<br><b>예상 동작</b> &nbsp; 각 문장의 예측·정답과 함께 "정확도: 5/5 = 100%" 같은 결과가 나오면 성공.
</div>
</div>

In [ ]:
# ✏️ 연습 1-7 · zero-shot 분류 + 정확도 채점
# (문장, 정답 라벨) 5개 — 학습 없이 프롬프트만으로 맞혀 봅니다. (s5-01 BERT는 '학습'으로 분류!)
samples = [
    ("이 영화 정말 최고였어요. 시간 가는 줄 몰랐네요.", "긍정"),
    ("돈이 아까웠습니다. 다신 안 볼 듯.", "부정"),
    ("연기도 좋고 음악도 좋아서 여운이 남아요.", "긍정"),
    ("스토리가 지루하고 결말도 뻔했어요.", "부정"),
    ("기대 이상! 주변에도 추천하고 싶어요.", "긍정"),
]

def classify(sentence):
    """문장을 받아 '긍정'/'부정' 한 단어만 반환."""
    # 🔧 직접 작성 (프롬프트) — sentence 를 긍정/부정 중 하나로만 답하게 하는 프롬프트
    prompt = ""   # 여기에 직접 작성
    return ____   # ask_with(SMALL_MODEL, prompt) 결과에서 앞뒤 공백을 없앤 값

# 🔧 직접 작성 (스켈레톤) — samples 를 돌며 예측을 정답과 비교하고 정확도를 구하세요.
correct = 0
for sentence, label in samples:
    pred = ____              # classify 로 sentence 예측
    if ____:                 # 예측이 정답(label)과 같으면
        correct += 1
    print(f"예측={pred} / 정답={label} | {sentence}")

accuracy = ____              # 맞은 수 / 전체 개수
print(f"정확도: {correct}/{len(samples)} = {accuracy:.0%}")


### Part 1 정리
- **명료한 지시 / 참고 텍스트 / 단계 분할 / 페르소나** — 모델은 그대로여도 프롬프트만으로 답이 크게 좋아집니다.
- 특히 **참고 텍스트 제공**은 다음 노트북에서 배울 **RAG** 의 출발점입니다.
- **프롬프트로 끝이 아니라 코드로 채점** — CoT 답 채점·JSON 파싱·zero-shot 분류 정확도처럼, 결과가 잘 나왔는지 **코드로 측정·검증**하는 게 실전 프롬프트 엔지니어링입니다.
- 이제 이 프롬프트들을 *매번 코드로 치지 않고* **챗봇 화면**에서 편하게 시험해 봅시다.
- **작은 모델일수록 프롬프트가 더 중요** — 좋은 프롬프트면 작은(저렴한) 모델로도 큰 모델 수준에 근접(= 비용 절감).

### 1-8. (다 같이) 내가 만든 프롬프트 · 결과 공유하기

프롬프트는 **직접 많이 써 봐야** 감이 옵니다. 위 원칙을 응용해 **여러분만의 프롬프트** 를 만들고, 결과를 **공용 시트** 에 올려 다 같이 봅시다.

**할 일**
1. 아래 `my_prompt` 를 여러 번 바꿔 실행해 보세요 (모호 vs 명료 · 페르소나 · 단계별 · 참고자료 등 배운 기법 응용).
2. 재밌거나 신기한 결과가 나오면 **공용 시트** 에 `프롬프트 | 결과 | 한 줄 코멘트` 로 적으세요.
   - 시트(수업 중 링크 공유): https://docs.google.com/spreadsheets/d/1UJvf4pHRCPGaXEWtP6w6IbAN0k-7iBUlr6fUYWRu9v4/edit
3. 다 같이 화면으로 보며, **어떤 프롬프트가 왜 좋은 결과를 냈는지** 이야기해 봅니다.

In [ ]:
# my_prompt 를 바꿔 실행 → 마음에 드는 결과는 공용 시트에 기록!
my_prompt = "카페 신메뉴 '흑임자 라떼'를 소개하는 인스타 문구를, 이모지 포함 3줄로 써줘."
ask(my_prompt)

# (아이디어) 아래처럼 기법을 바꿔가며 같은 주제를 여러 번 시도해 보세요.
# - 페르소나: "너는 20년차 카피라이터야. ..."
# - 제약 추가: "10자 이내 한 문장, 이모지 금지."
# - 단계 요구: "먼저 타깃 고객을 정하고, 그에 맞춰 문구 3안을 제시해줘."

---
# Part 2 · 나만의 Gradio 챗봇 만들기

지금까지는 셀에서 한 번씩 질문했습니다. 이제 **웹 채팅 화면**을 띄워서, 사람처럼 *대화를 주고받는* 챗봇을 만들어 봅니다.

도구는 **Gradio** 입니다. 함수 하나만 정의하면 채팅 UI를 자동으로 만들어 줍니다.
핵심 개념 두 가지:
- **시스템 프롬프트(system prompt)**: 챗봇의 *성격·역할·규칙* 을 정하는 숨은 지시문. 사용자에게는 안 보이지만 모든 답변에 영향을 줍니다. = **페르소나의 핵심**.
- **대화 기록(history)**: 앞에서 무슨 말을 했는지 모델에게 같이 넘겨야 *맥락을 기억하는* 대화가 됩니다.

## 2-1. 가장 간단한 챗봇 (입력을 그대로 따라 말하기)
먼저 AI 없이 **Gradio 채팅창이 어떻게 동작하는지** 감을 잡습니다.
`gr.ChatInterface` 는 우리가 만든 `response` 함수를 호출해 답을 화면에 보여줍니다.

- `message` : 사용자가 방금 친 말
- `history` : 지금까지의 대화 기록 (`{"role": "user"/"assistant", "content": ...}` 형태의 목록)

> 실행하면 셀 아래에 채팅창이 뜹니다. Colab에서는 **공개 URL(`https://...gradio.live`)** 도 함께 생성됩니다(1주일간 유효).
> 다 써 본 뒤에는 **다음 챗봇을 켜기 전에** 바로 아래의 `demo.close()` 셀을 실행해 이전 창을 꺼 주세요(포트 충돌 방지).

In [ ]:
import gradio as gr

def echo_response(message, history):
    # AI를 부르지 않고, 받은 말을 그대로 따라 합니다.
    return f"당신이 말한 내용: {message}"

demo = gr.ChatInterface(
    fn=echo_response,
    title="따라 말하기 챗봇",
    description="아직 AI가 아니라, 입력을 그대로 돌려주는 연습용 챗봇입니다.",
)
demo.launch()

In [ ]:
# 위 챗봇을 다 써 봤다면 이 셀을 실행해 창을 닫습니다(다음 챗봇 실행 전 권장).
demo.close()

### 2-1½. 싱글턴 vs 멀티턴 — 왜 '대화 기억'이 필요할까

- **싱글턴(single-turn)**: 질문 하나 → 답 하나. 이전 대화를 **기억 못 함.**
- **멀티턴(multi-turn)**: 이전 대화를 **함께 넘겨** 맥락을 이어감(진짜 대화처럼).

아래에서 **기억 없이(싱글턴)** 이어지는 질문을 던져, 모델이 앞 내용을 못 잇는 걸 직접 확인해 봅시다.

In [ ]:
from langchain_openai import ChatOpenAI
llm = ChatOpenAI(model="gpt-4o-mini")

# 싱글턴 - 매 질문을 '따로' 보냄 (이전 대화를 안 넘김)
print("Q1: 내 이름은 지민이야. 기억해줘.")
print("A1:", llm.invoke("내 이름은 지민이야. 기억해줘.").content)
print()
print("Q2: 방금 내 이름이 뭐라고 했지?")
print("A2:", llm.invoke("방금 내 이름이 뭐라고 했지?").content)   # 앞 대화를 안 넘겨 이름을 모름!

→ **A2에서 이름을 못 맞히죠?** 이전 대화를 안 넘겼기 때문입니다.
**그럼 어떻게 고칠까요?** 답은 간단합니다 — **이전 대화를 리스트에 쌓아 매번 함께 넘기면** 됩니다. 바로 해봅시다.

## 2-2. 멀티턴으로 고치기 — 대화를 리스트에 쌓아 넘기기

`ChatOpenAI` 는 문자열 하나 대신 **메시지 리스트** 를 받을 수 있어요. `HumanMessage`(사용자)·`AIMessage`(모델의 답) 를 **대화 순서대로 쌓아** 넘기면 모델이 앞 내용을 봅니다.

먼저 "리스트로도 부를 수 있다"는 것만 확인하고(아래 셀), **앞 대화를 기억하게 만드는 건 여러분이 직접** 구현합니다.

In [ ]:
from langchain_core.messages import HumanMessage, AIMessage

# 문자열 대신 '메시지 리스트' 로도 모델을 부를 수 있다 (아직은 1턴, 기억 없음)
msgs = [HumanMessage(content="안녕! 한 문장으로 자기소개 해줘.")]
print(llm.invoke(msgs).content)

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 연습 2-2 · 멀티턴(대화 기억) 직접 구현<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>문제</b> &nbsp; <code>messages</code> 리스트에 사람/AI 메시지를 <b>번갈아 쌓아</b>, 모델이 앞 대화를 기억하게 만드세요. 최소 2턴(정보 알려주기 → 그 정보 되묻기).<br><b>힌트</b> &nbsp; ① <code>HumanMessage</code> 로 첫 말을 담은 리스트를 만들고 <code>llm.invoke</code> → 답을 얻는다. ② 그 답을 <code>AIMessage</code> 로 리스트에 <b>append</b> 하고, 다음 질문을 <code>HumanMessage</code> 로 append 한 뒤 다시 <code>invoke</code>. (<code>append</code> 는 3차시에서 배운 리스트 메서드!)<br><b>예상 동작</b> &nbsp; 두 번째 답에서 모델이 앞에서 알려준 정보를 <b>기억</b>해 답하면 성공.
</div>
</div>

In [ ]:
# 🔧 직접 작성 (스켈레톤) — 변수 흐름은 주어졌습니다. 빈 곳을 채워 '기억하는 대화'를 완성하세요.
# 목표: 1) 내 이름/좋아하는 것 등을 알려주고  2) 그걸 기억하는지 되물어 확인
# (필요한 건 위에서 import 한 HumanMessage · AIMessage · llm)

messages = [HumanMessage(content="내 이름은 지민이야. 그리고 내가 제일 좋아하는 음식은 떡볶이야.")]
answer1 = ____                 # messages 를 llm 에 넘겨(.invoke) 답의 .content 얻기
print("A1:", answer1)

____                           # 방금 받은 answer1 을 AIMessage 로 messages 에 추가(append)
____                           # 다음 질문("방금 말한 내 이름과 좋아하는 음식이 뭐였지?")을 HumanMessage 로 추가
answer2 = ____                 # 다시 llm 에 messages 를 넘겨 답의 .content 얻기
print("A2:", answer2)


## 2-3. 시스템 프롬프트 한 조각 더 — 성격 입히기

멀티턴이 됐으니, 이제 **리스트 맨 앞에 `SystemMessage`** 를 하나 얹어 챗봇의 *성격·역할·규칙* 을 정합니다.
사용자에겐 안 보이는 **숨은 지시문 = 페르소나** 예요. 맨 앞에 넣기만 하면 모든 답에 영향을 줍니다.

In [ ]:
from langchain_core.messages import SystemMessage

# 🔧 직접 해보기: SYSTEM_PROMPT 를 원하는 성격으로 바꿔 실행해 보세요.
SYSTEM_PROMPT = "너는 밝고 명랑한 초등학교 선생님이야. 쉽고 짧게, 존댓말로 답해줘."   # TODO: 성격 바꾸기

messages = [
    SystemMessage(content=SYSTEM_PROMPT),
    HumanMessage(content="딥러닝이 뭐야?"),
]
print(llm.invoke(messages).content)

성격(시스템 프롬프트)을 바꾸면 **같은 질문** 이라도 말투·눈높이가 달라집니다.
이제 지금까지 만든 두 조각 — **① 시스템 프롬프트 · ② history 쌓기** — 를 함수 하나로 묶어 **채팅 화면(Gradio)** 에 연결합니다.

## 2-4. Gradio 챗봇으로 합치기 (시스템 프롬프트 + 대화 기억)

이제 배운 두 조각 — **① 시스템 프롬프트 · ② history 쌓기** — 를 `chat_response` 함수 **하나에 직접 합칩니다.** `gr.ChatInterface` 는 지금까지의 대화(`history`)를 **자동으로 쌓아** 매 호출에 넘겨주니, 우리는 그 history 를 메시지 리스트로 바꿔 모델에 넘기기만 하면 됩니다.

> 실행하면 셀 아래에 채팅창이 뜹니다. Colab에서는 공개 URL(`https://...gradio.live`)도 생성됩니다.
> **다음 챗봇을 켜기 전** 아래 `demo.close()` 를 실행해 이전 창을 꺼 주세요(포트 충돌 방지).

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">🔧 실습 2-4 · 챗봇 본체 <code>chat_response</code> 직접 작성<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>덱에서 본 원리</b> &nbsp; 이번엔 '시스템 프롬프트 + 대화 기억'을 함수 하나로 합쳐 진짜 챗봇으로.<br><b>문제</b> &nbsp; <code>gr.ChatInterface</code> 에 연결될 <code>chat_response(message, history)</code> 의 <b>본체</b>를 직접 작성하세요. 2-2(메시지 쌓기)와 2-3(시스템 프롬프트)에서 만든 조각을 합치면 됩니다.<br><b>힌트</b> &nbsp; <code>history</code> 는 <code>{"role": "user"/"assistant", "content": ...}</code> 형태의 목록입니다. 맨 앞 <code>SystemMessage</code> → <code>history</code> 를 순서대로 → 이번 <code>message</code> 순으로 메시지 리스트를 만들어 <code>chat_llm</code> 에 넘기세요.<br><b>예상 동작</b> &nbsp; 실행하면 채팅창이 뜨고, "방금 내가 뭐라고 했지?" 에 앞 대화를 <b>기억</b>해 답하면 성공.
</div>
</div>

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

# 챗봇의 성격(시스템 프롬프트) — 원하는 대로 바꿔도 됩니다.
SYSTEM_PROMPT = "너는 친절하고 상냥한 AI 비서야. 사용자의 말에 공감하며 한국어로 답해줘."

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def chat_response(message, history):
    # 🔧 직접 작성 (스켈레톤) — 2-2(대화 쌓기) + 2-3(시스템 프롬프트) 조각을 합치세요.
    messages = [____]                 # 맨 앞에 SystemMessage(성격) 하나 넣기
    for turn in history:              # 지난 대화(사용자/AI)를 순서대로 이어 붙이기
        if turn["role"] == "user":
            messages.append(____)     # 사용자 발화 → HumanMessage 로 감싸기
        else:
            messages.append(____)     # AI 발화 → AIMessage 로 감싸기
    messages.append(____)             # 이번에 받은 message → HumanMessage 로 감싸 마지막에 추가
    return ____                       # 완성된 messages 를 chat_llm 에 넘겨 답의 .content 반환

demo = gr.ChatInterface(
    fn=chat_response,
    title="나만의 AI 챗봇",
    description="시스템 프롬프트로 성격을 정하고, 대화를 기억하는 챗봇입니다.",
    examples=["안녕! 오늘 기분이 좀 우울해.", "방금 내가 뭐라고 했지?", "주말에 할 만한 취미 추천해줘."],
)
demo.launch()


In [ ]:
demo.close()

## 2-5. 성격 바꿔 보기 — 화면에서 시스템 프롬프트 수정
매번 코드를 고치지 않고, **채팅 화면에서 시스템 프롬프트를 직접 입력** 해 성격을 바꿔 봅시다.
`additional_inputs` 로 입력칸을 하나 더 달면, 그 값이 `persona_response` 의 세 번째 인자로 들어옵니다.

🔧 **직접 해보기** — 채팅창 아래 'Additional Inputs' 를 펼쳐 시스템 프롬프트를 바꾸고 다시 말을 걸어 보세요. 예:
- `너는 8살 아이야. 아이처럼 신나고 짧게 말해줘.`
- `너는 츤데레 고양이 집사야. 살짝 퉁명스럽지만 결국 도와줘.`
- `너는 깐깐한 면접관이야. 짧고 날카롭게 되묻는다.`

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.8)

def persona_response(message, history, system_prompt):
    sp = system_prompt.strip() or "너는 친절한 AI 비서야. 한국어로 답해줘."
    messages = [SystemMessage(content=sp)]
    for turn in history:
        role = HumanMessage if turn["role"] == "user" else AIMessage
        messages.append(role(content=turn["content"]))
    messages.append(HumanMessage(content=message))
    return chat_llm.invoke(messages).content

demo = gr.ChatInterface(
    fn=persona_response,
    title="성격을 바꿀 수 있는 챗봇",
    description="아래 시스템 프롬프트를 바꾸면 챗봇의 성격이 달라집니다.",
    additional_inputs=[
        gr.Textbox(value="너는 8살 아이야. 아이처럼 신나고 짧게 말해줘.",
                   label="시스템 프롬프트 (챗봇의 성격)", lines=3)
    ],
)
demo.launch()

In [ ]:
demo.close()

<div style="border:1px solid #E7DCC6;border-left:6px solid #C15F3C;border-radius:10px;background:#FBF4EA;padding:16px 20px;margin:12px 0;">
<div style="font-size:16px;font-weight:700;color:#A8472A;margin-bottom:8px;">✏️ 실습 과제 · 나만의 자기소개서 도우미 챗봇<span style="font-size:12px;color:#A8472A;background:#F0DCCF;padding:1px 8px;border-radius:10px;margin-left:6px;">직접 작성</span></div>
<div style="color:#3D3A33;line-height:1.8;font-size:14.5px;">
<b>미션</b> &nbsp; 배운 조각(시스템 프롬프트 + 멀티턴)을 모아 아래 3가지를 <code>MY_SYSTEM_PROMPT</code> 에 녹이세요.<br>&nbsp;&nbsp;① <b>역할·말투</b> — "너는 취업 자기소개서 컨설턴트야. 존댓말로, 지원자를 격려하며 돕는다."<br>&nbsp;&nbsp;② <b>멀티턴 인터뷰</b> — 한 번에 다 묻지 말고 <b>한 항목씩</b>: 지원 직무 → 강점 1가지 → 근거 경험 → 마무리.<br>&nbsp;&nbsp;③ <b>출력 형식</b> — 모은 답으로 <b>3~4문장 초안</b> + <b>개선점 1가지</b>.<br><b>힌트</b> &nbsp; 위 ①~③을 그대로 문장으로 풀어 적으면 됩니다(코드는 이미 완성). 실행 후 챗봇의 질문에 답해 초안이 나오는지 확인하세요.<br><b>관찰 포인트</b> &nbsp; 챗봇이 <b>한 항목씩</b> 물어보고, 마지막에 초안+개선점을 주면 성공.
</div>
</div>

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
import gradio as gr

# TODO: 아래 따옴표 사이에 '자기소개서 도우미' 시스템 프롬프트를 직접 작성하세요.
#  (힌트: 위 미션 1~3의 역할·질문방식·포함항목·말투를 문장으로 적으면 됩니다.)
MY_SYSTEM_PROMPT = """
(여기에 챗봇의 역할·질문 방식·포함 항목·말투를 적으세요)
"""

chat_llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)

def resume_response(message, history):
    sp = MY_SYSTEM_PROMPT.strip() or "너는 자기소개서 작성을 도와주는 컨설턴트야. 한국어 존댓말로 답해줘."
    messages = [SystemMessage(content=sp)]
    for turn in history:
        role = HumanMessage if turn["role"] == "user" else AIMessage
        messages.append(role(content=turn["content"]))
    messages.append(HumanMessage(content=message))
    return chat_llm.invoke(messages).content

demo = gr.ChatInterface(
    fn=resume_response,
    title="나만의 자기소개서 챗봇",
    description="질문에 답하다 보면 자기소개서 초안이 만들어집니다.",
    examples=["자기소개서 작성을 도와줘.", "내가 인상 깊게 들은 과목은 '머신러닝 입문'이야.", "방금 문단을 더 겸손한 톤으로 다시 써줘."],
)
demo.launch()

In [ ]:
demo.close()

## 🔑 막히면 펼쳐보기 (힌트)

> 완성 코드가 아니라 **접근·메서드 힌트**입니다. 조각을 참고해 **자기 코드에 맞춰 조립**하세요.

<details>
<summary>compare() 가 막히면</summary>

- `ask(a)` 처럼 <b>ask 를 그대로 호출</b>하면 출력까지 됩니다. a·b 를 각각 한 번씩 부르면 끝.
</details>

<details>
<summary>페르소나 for 가 막히면</summary>

- 모델에 줄 프롬프트 = <code>f"{persona}\n질문: {question}"</code> 형태의 한 문자열.
- 그 문자열을 <code>ask(...)</code> 에 넘기면 출력됩니다.
</details>

<details>
<summary>CoT 채점(re) 이 막히면</summary>

- 숫자만 뽑기: <code>re.findall(r"-?\d+", text)</code> → 정수 문자열 리스트.
- 최종 답 = 그 리스트의 <b>마지막 값</b>을 <code>int()</code> 로.
- 판정: 뽑은 값이 <code>correct_answer</code> 와 <b>같은지</b> 비교.
</details>

<details>
<summary>JSON 파싱이 막히면</summary>

- 프롬프트에 <b>"설명·코드블록 없이 JSON만 출력"</b> 을 꼭 넣기.
- 문자열 → dict: <code>json.loads(reply)</code>.
- 값 꺼내기: <code>data["sentiment"]</code>, <code>data["reason"]</code>.
- 파싱 에러가 나면 프롬프트를 더 강하게 고쳐 재실행.
</details>

<details>
<summary>zero-shot 채점 루프가 막히면</summary>

- classify 프롬프트: "<b>'긍정' 또는 '부정' 한 단어로만</b> 답해" + 문장 포함.
- 답 정리: <code>ask_with(...).strip()</code>.
- 정확도: 맞은 개수를 세는 <code>correct</code> 를 두고, 마지막에 <code>correct / len(samples)</code>.
</details>

<details>
<summary>멀티턴 루프가 막히면</summary>

- 핵심 순서: invoke → 답을 <code>AIMessage</code> 로 append → 다음 질문을 <code>HumanMessage</code> 로 append → 다시 invoke.
- 호출·답 얻기: <code>llm.invoke(messages).content</code>.
</details>

<details>
<summary>chat_response 본체가 막히면</summary>

- 리스트 시작: <code>[SystemMessage(content=SYSTEM_PROMPT)]</code>.
- history 각 turn: role 이 "user" 면 <code>HumanMessage</code>, 아니면 <code>AIMessage</code> 로 <code>turn["content"]</code> 를 감싸 append.
- 마지막에 이번 <code>message</code> 를 <code>HumanMessage</code> 로 append.
- 반환: <code>chat_llm.invoke(messages).content</code>.
</details>


---
## 마무리

이 노트북에서 한 것:
- **프롬프트 엔지니어링 4원칙**(명료한 지시 · 참고 텍스트 · 작업 분할 · 페르소나)을 직접 비교 실습.
- **Gradio 챗봇**을 만들고, **시스템 프롬프트(페르소나)** 와 **대화 기록** 으로 성격 있는 대화를 구현.

**다음 노트북 예고 — RAG 챗봇**: 1-2에서 *참고 텍스트를 직접 붙여넣어* 답하게 했던 것을, **여러 문서에서 자동으로 관련 부분을 찾아** 답하도록 확장합니다. "내 문서로 답하는 챗봇"을 만들어 봅시다.